# DINOv3 - régression de calories

Fine-tuning d'un DINOv3 pré-entraîné (Hugging Face) pour prédire le nombre de calories d'un plat à partir d'une photo.
Notebook exécuté sur Kaggle (GPU) dans le cadre de la compétition *M2 Food Calorie Estimation*.

**Démarche :**
- Cible transformée en **log** puis standardisée (distribution très asymétrique à droite), MAE mesurée en kcal.
- Augmentation de données forte, learning rates différenciés backbone/tête, précision mixte (AMP), scheduler cosine avec warmup.
- Validation croisée 5 folds, TTA (flip horizontal) puis moyenne des 5 modèles pour la prédiction finale.

**Résultat : CV MAE = 39.6 ± 3.8 kcal.**

Les sorties affichées sont celles du run complet (ViT-B/16, 5 folds, 30 epochs).

In [ ]:
# DINOv3 est un modèle "gated" sur Hugging Face : il faut accepter la licence et s'authentifier avec un token.
# https://huggingface.co/facebook/dinov3-vitb16-pretrain-lvd1689m
!pip install -q --upgrade-strategy only-if-needed "transformers>=4.56"

from huggingface_hub import login
login(CENSURED)

## Configuration

Backbone **DINOv3 ViT-B/16** (patch de 16 px) avec des images en 480×480. Le backbone pré-entraîné est affiné avec un learning rate
100 fois plus faible que celui de la tête de régression, pour ne pas détruire ses représentations.

In [ ]:
import os, random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import KFold
from transformers import AutoModel, get_cosine_schedule_with_warmup

class CFG:
    data_dir   = "/kaggle/input/competitions/m2-food-calorie-estimation"
    train_csv  = data_dir + "/train_labels.csv"
    test_csv   = data_dir + "/test_ids.csv"
    train_img  = data_dir + "/train/images"
    test_img   = data_dir + "/test/images"

    model_name = "facebook/dinov3-vitb16-pretrain-lvd1689m"
    img_size   = 480          # Multiples de 16 pour DINOv3 (patch size = 16)
    n_folds    = 5
    epochs     = 30
    batch_size = 16
    lr_backbone  = 1e-5
    lr_head      = 1e-3
    weight_decay = 0.05
    warmup_ratio = 0.1
    num_workers  = 2
    grad_clip    = 1.0
    seed = 42
    device = "cuda" if torch.cuda.is_available() else "cpu"
    out_csv = "submission.csv"

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

print(f"Device: {CFG.device} | Modèle: {CFG.model_name}")

Device: cuda | Modèle: facebook/dinov3-vitb16-pretrain-lvd1689m


## Données et augmentations

- **Entraînement** : recadrage aléatoire (60 à 100 % de l'image), flips horizontal et vertical, rotation de ±15° et variations de couleur.
  Les photos de plats sont souvent prises du dessus, donc ces transformations ne changent pas le contenu calorique.
- **Validation / test** : redimensionnement puis recadrage central.
- Normalisation ImageNet.

In [3]:
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

def build_transforms(train):
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(CFG.img_size, scale=(0.6, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(0.3, 0.3, 0.3, 0.05),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
        ])
    return transforms.Compose([
        transforms.Resize(int(CFG.img_size * 1.14)),
        transforms.CenterCrop(CFG.img_size),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])

class FoodDataset(Dataset):
    """Renvoie (image, cible_log_standardisee). cible = -1 si test."""
    def __init__(self, df, img_dir, tfm, targets=None):
        self.files = df["filename"].tolist()
        self.dir = img_dir
        self.tfm = tfm
        self.targets = targets
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        img = Image.open(os.path.join(self.dir, self.files[i])).convert("RGB")
        img = self.tfm(img)
        y = self.targets[i] if self.targets is not None else -1.0
        return img, torch.tensor(y, dtype=torch.float32)

## Modèle

La sortie du backbone est résumée par la concaténation de deux vecteurs :
- le **token CLS**, qui donne une représentation globale de l'image ;
- la **moyenne des patch tokens**, plus sensible au détail et à la quantité de nourriture.

Les *register tokens* de DINOv3 sont exclus de la moyenne. Une petite tête MLP (LayerNorm → 512 → 1) prédit la cible standardisée.

In [ ]:
class CalorieRegressor(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        dim = self.backbone.config.hidden_size

        n_reg = getattr(self.backbone.config, "num_register_tokens", 0)
        self.n_prefix = 1 + n_reg

        # Pooling = CLS (global) concatene a la moyenne des patch tokens (dense / fine-grained)
        self.head = nn.Sequential(
            nn.LayerNorm(dim * 2),
            nn.Linear(dim * 2, 512), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(512, 1),
        )
    def forward(self, x):
        out = self.backbone(pixel_values=x).last_hidden_state
        cls = out[:, 0]                          # token CLS
        patches = out[:, self.n_prefix:].mean(1) # moyenne des patch tokens (hors registers)
        feat = torch.cat([cls, patches], dim=-1)
        return self.head(feat).squeeze(-1)

## Entraînement d'un fold

- Loss **Smooth L1** (Huber) sur la cible log-standardisée, moins sensible aux plats extrêmes que la MSE.
- AdamW, scheduler cosine avec 10 % de warmup, gradient clipping, précision mixte.
- À chaque epoch, la MAE de validation est calculée **en kcal** (après retour à l'échelle d'origine) et on garde les poids de la meilleure epoch.
- Prédiction sur le test avec **TTA** : moyenne de la prédiction sur l'image et sur son miroir horizontal.

In [5]:
def to_kcal(pred_std, mu, sigma):
    return np.expm1(pred_std * sigma + mu)

def train_fold(tr_df, va_df, test_df, mu, sigma, fold):
    set_seed(CFG.seed + fold)
    tr_ds = FoodDataset(tr_df, CFG.train_img, build_transforms(True),  tr_df["y"].values)
    va_ds = FoodDataset(va_df, CFG.train_img, build_transforms(False), va_df["y"].values)
    tr_dl = DataLoader(tr_ds, CFG.batch_size, shuffle=True,  num_workers=CFG.num_workers, pin_memory=True, drop_last=True)
    va_dl = DataLoader(va_ds, CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=True)

    model = CalorieRegressor(CFG.model_name).to(CFG.device)
    optim = torch.optim.AdamW([
        {"params": model.backbone.parameters(), "lr": CFG.lr_backbone},
        {"params": model.head.parameters(),     "lr": CFG.lr_head},
    ], weight_decay=CFG.weight_decay)
    steps = len(tr_dl) * CFG.epochs
    sched = get_cosine_schedule_with_warmup(optim, int(steps * CFG.warmup_ratio), steps)
    loss_fn = nn.SmoothL1Loss()
    scaler = torch.cuda.amp.GradScaler()

    best_mae, best_state = 1e9, None
    for epoch in range(CFG.epochs):
        model.train()
        for x, y in tr_dl:
            x, y = x.to(CFG.device), y.to(CFG.device)
            optim.zero_grad()
            with torch.cuda.amp.autocast():
                loss = loss_fn(model(x), y)
            scaler.scale(loss).backward()
            scaler.unscale_(optim)
            nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
            scaler.step(optim); scaler.update(); sched.step()

        model.eval(); preds, gts = [], []
        with torch.no_grad(), torch.cuda.amp.autocast():
            for x, y in va_dl:
                preds.append(model(x.to(CFG.device)).float().cpu().numpy())
                gts.append(y.numpy())
        preds = to_kcal(np.concatenate(preds), mu, sigma)
        gts   = to_kcal(np.concatenate(gts), mu, sigma)
        mae = np.mean(np.abs(preds - gts))
        print(f"[fold {fold}] epoch {epoch+1}/{CFG.epochs}  val MAE = {mae:.2f}")
        if mae < best_mae:
            best_mae = mae
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    te_ds = FoodDataset(test_df, CFG.test_img, build_transforms(False))
    te_dl = DataLoader(te_ds, CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=True)
    model.eval(); test_pred = []
    with torch.no_grad(), torch.cuda.amp.autocast():
        for x, _ in te_dl:
            x = x.to(CFG.device)
            p = model(x).float()
            p_flip = model(torch.flip(x, dims=[3])).float()
            test_pred.append(((p + p_flip) / 2).cpu().numpy())
    test_pred = to_kcal(np.concatenate(test_pred), mu, sigma)
    del model; torch.cuda.empty_cache()
    return best_mae, test_pred

## Validation croisée et soumission

5 folds : chaque fold entraîne un modèle et prédit le test. La soumission finale est la moyenne des 5 prédictions.

In [6]:
set_seed(CFG.seed)
train_df = pd.read_csv(CFG.train_csv)
test_df  = pd.read_csv(CFG.test_csv)

logc = np.log1p(train_df["calories"].values)
mu, sigma = logc.mean(), logc.std()
train_df["y"] = (logc - mu) / sigma

kf = KFold(CFG.n_folds if CFG.n_folds > 1 else 5, shuffle=True, random_state=CFG.seed)
maes, test_preds = [], []
for fold, (tr_idx, va_idx) in enumerate(kf.split(train_df)):
    mae, tp = train_fold(
        train_df.iloc[tr_idx].reset_index(drop=True),
        train_df.iloc[va_idx].reset_index(drop=True),
        test_df, mu, sigma, fold)
    maes.append(mae); test_preds.append(tp)
    if CFG.n_folds == 1:
        break

print(f"\nCV MAE = {np.mean(maes):.2f} +/- {np.std(maes):.2f}")

final = np.clip(np.mean(test_preds, axis=0), 1.0, None)
sub = pd.DataFrame({"image_id": test_df["image_id"], "predicted_calories": final})
sub.to_csv(CFG.out_csv, index=False)
print(f"Soumission ecrite -> {CFG.out_csv}  ({len(sub)} lignes)")
sub.head()

[fold 0] epoch 1/30  val MAE = 102.84
[fold 0] epoch 2/30  val MAE = 96.26
[fold 0] epoch 3/30  val MAE = 84.52
[fold 0] epoch 4/30  val MAE = 73.02
[fold 0] epoch 5/30  val MAE = 71.52
[fold 0] epoch 6/30  val MAE = 67.70
[fold 0] epoch 7/30  val MAE = 91.40
[fold 0] epoch 8/30  val MAE = 55.54
[fold 0] epoch 9/30  val MAE = 72.40
[fold 0] epoch 10/30  val MAE = 52.02
[fold 0] epoch 11/30  val MAE = 59.60
[fold 0] epoch 12/30  val MAE = 57.72
[fold 0] epoch 13/30  val MAE = 72.15
[fold 0] epoch 14/30  val MAE = 44.78
[fold 0] epoch 15/30  val MAE = 53.91
[fold 0] epoch 16/30  val MAE = 47.37
[fold 0] epoch 17/30  val MAE = 48.69
[fold 0] epoch 18/30  val MAE = 44.88
[fold 0] epoch 19/30  val MAE = 46.25
[fold 0] epoch 20/30  val MAE = 44.00
[fold 0] epoch 21/30  val MAE = 44.35
[fold 0] epoch 22/30  val MAE = 39.21
[fold 0] epoch 23/30  val MAE = 42.32
[fold 0] epoch 24/30  val MAE = 47.75
[fold 0] epoch 25/30  val MAE = 43.22
[fold 0] epoch 26/30  val MAE = 39.67
[fold 0] epoch 27/30

[fold 1] epoch 1/30  val MAE = 149.78
[fold 1] epoch 2/30  val MAE = 114.80
[fold 1] epoch 3/30  val MAE = 109.46
[fold 1] epoch 4/30  val MAE = 104.33
[fold 1] epoch 5/30  val MAE = 111.88
[fold 1] epoch 6/30  val MAE = 74.30
[fold 1] epoch 7/30  val MAE = 91.27
[fold 1] epoch 8/30  val MAE = 73.73
[fold 1] epoch 9/30  val MAE = 62.73
[fold 1] epoch 10/30  val MAE = 68.53
[fold 1] epoch 11/30  val MAE = 54.77
[fold 1] epoch 12/30  val MAE = 52.07
[fold 1] epoch 13/30  val MAE = 65.50
[fold 1] epoch 14/30  val MAE = 61.19
[fold 1] epoch 15/30  val MAE = 51.95
[fold 1] epoch 16/30  val MAE = 56.48
[fold 1] epoch 17/30  val MAE = 51.36
[fold 1] epoch 18/30  val MAE = 51.12
[fold 1] epoch 19/30  val MAE = 45.95
[fold 1] epoch 20/30  val MAE = 48.13
[fold 1] epoch 21/30  val MAE = 44.55
[fold 1] epoch 22/30  val MAE = 42.25
[fold 1] epoch 23/30  val MAE = 45.77
[fold 1] epoch 24/30  val MAE = 47.26
[fold 1] epoch 25/30  val MAE = 46.88
[fold 1] epoch 26/30  val MAE = 40.75
[fold 1] epoch 2

[fold 2] epoch 1/30  val MAE = 125.08
[fold 2] epoch 2/30  val MAE = 155.83
[fold 2] epoch 3/30  val MAE = 85.46
[fold 2] epoch 4/30  val MAE = 96.77
[fold 2] epoch 5/30  val MAE = 74.02
[fold 2] epoch 6/30  val MAE = 72.28
[fold 2] epoch 7/30  val MAE = 91.68
[fold 2] epoch 8/30  val MAE = 88.81
[fold 2] epoch 9/30  val MAE = 66.08
[fold 2] epoch 10/30  val MAE = 71.17
[fold 2] epoch 11/30  val MAE = 77.80
[fold 2] epoch 12/30  val MAE = 80.90
[fold 2] epoch 13/30  val MAE = 62.97
[fold 2] epoch 14/30  val MAE = 61.06
[fold 2] epoch 15/30  val MAE = 53.67
[fold 2] epoch 16/30  val MAE = 58.97
[fold 2] epoch 17/30  val MAE = 62.76
[fold 2] epoch 18/30  val MAE = 58.12
[fold 2] epoch 19/30  val MAE = 63.47
[fold 2] epoch 20/30  val MAE = 52.67
[fold 2] epoch 21/30  val MAE = 50.60
[fold 2] epoch 22/30  val MAE = 55.47
[fold 2] epoch 23/30  val MAE = 53.51
[fold 2] epoch 24/30  val MAE = 48.05
[fold 2] epoch 25/30  val MAE = 47.12
[fold 2] epoch 26/30  val MAE = 46.84
[fold 2] epoch 27/3

[fold 3] epoch 1/30  val MAE = 116.85
[fold 3] epoch 2/30  val MAE = 107.77
[fold 3] epoch 3/30  val MAE = 78.14
[fold 3] epoch 4/30  val MAE = 85.65
[fold 3] epoch 5/30  val MAE = 73.73
[fold 3] epoch 6/30  val MAE = 66.11
[fold 3] epoch 7/30  val MAE = 78.21
[fold 3] epoch 8/30  val MAE = 64.68
[fold 3] epoch 9/30  val MAE = 56.07
[fold 3] epoch 10/30  val MAE = 62.86
[fold 3] epoch 11/30  val MAE = 54.28
[fold 3] epoch 12/30  val MAE = 57.71
[fold 3] epoch 13/30  val MAE = 62.70
[fold 3] epoch 14/30  val MAE = 49.91
[fold 3] epoch 15/30  val MAE = 45.31
[fold 3] epoch 16/30  val MAE = 53.87
[fold 3] epoch 17/30  val MAE = 48.24
[fold 3] epoch 18/30  val MAE = 51.95
[fold 3] epoch 19/30  val MAE = 46.98
[fold 3] epoch 20/30  val MAE = 48.20
[fold 3] epoch 21/30  val MAE = 45.24
[fold 3] epoch 22/30  val MAE = 42.47
[fold 3] epoch 23/30  val MAE = 41.52
[fold 3] epoch 24/30  val MAE = 39.19
[fold 3] epoch 25/30  val MAE = 37.50
[fold 3] epoch 26/30  val MAE = 38.70
[fold 3] epoch 27/3

[fold 4] epoch 1/30  val MAE = 194.93
[fold 4] epoch 2/30  val MAE = 105.48
[fold 4] epoch 3/30  val MAE = 104.28
[fold 4] epoch 4/30  val MAE = 90.89
[fold 4] epoch 5/30  val MAE = 78.35
[fold 4] epoch 6/30  val MAE = 82.14
[fold 4] epoch 7/30  val MAE = 71.76
[fold 4] epoch 8/30  val MAE = 78.93
[fold 4] epoch 9/30  val MAE = 61.37
[fold 4] epoch 10/30  val MAE = 67.43
[fold 4] epoch 11/30  val MAE = 69.14
[fold 4] epoch 12/30  val MAE = 71.15
[fold 4] epoch 13/30  val MAE = 48.03
[fold 4] epoch 14/30  val MAE = 46.01
[fold 4] epoch 15/30  val MAE = 49.74
[fold 4] epoch 16/30  val MAE = 46.72
[fold 4] epoch 17/30  val MAE = 46.09
[fold 4] epoch 18/30  val MAE = 55.47
[fold 4] epoch 19/30  val MAE = 47.98
[fold 4] epoch 20/30  val MAE = 42.21
[fold 4] epoch 21/30  val MAE = 42.56
[fold 4] epoch 22/30  val MAE = 38.86
[fold 4] epoch 23/30  val MAE = 44.93
[fold 4] epoch 24/30  val MAE = 39.25
[fold 4] epoch 25/30  val MAE = 37.75
[fold 4] epoch 26/30  val MAE = 40.19
[fold 4] epoch 27/

,image_id,predicted_calories
0,test_0000,97.062183
1,test_0001,165.919875
2,test_0002,102.566211
3,test_0003,269.491695
4,test_0004,220.659052


## Résultats

| Fold | 0 | 1 | 2 | 3 | 4 | **Moyenne** |
|---|---|---|---|---|---|---|
| Meilleure MAE de validation (kcal) | 38.3 | 40.4 | 46.5 | 37.5 | 35.2 | **39.6 ± 3.8** |

- Les scores sont proches d'un fold à l'autre : le modèle est stable.
- La MAE de validation diminue encore en fin d'entraînement, sans signe de surapprentissage sur 30 epochs.
- La MAE retenue est celle de la meilleure epoch sur chaque fold. Elle est donc légèrement optimiste : en fin d'entraînement,
  la MAE varie d'environ ±3 kcal d'une epoch à l'autre.